# Lab 10 · Multi-GPU · MPI + CUDA · one rank per GPU, one GPU per rank

Combine MPI (from lab 06) with CUDA (from lab 09): each rank pins itself to one GPU, halos exchange through **CUDA-aware MPI** without copying to host memory. This is how every serious HPC AI/simulation code scales in 2026.

**Prerequisites.** Lab 06 (MPI halo pattern), Lab 09 (CUDA kernel). On Polaris (4 GPUs per node).

**Builds toward.** Lab 11 (large-scale scaling study using this hybrid stack).

> **📚 Where to look when you're stuck**
>
> - [**CUDA-aware MPI**](https://developer.nvidia.com/blog/introduction-cuda-aware-mpi/) — Nvidia's intro
> - [**Polaris multi-GPU jobs**](https://docs.alcf.anl.gov/polaris/running-jobs/)
> - [**NCCL** (alternative to MPI for GPU-GPU comm)](https://developer.nvidia.com/nccl)



## How this notebook works

Same three surfaces as prior labs: **[Hub]**, **[Hub -> cluster]**, **[cluster compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab10", host="polaris",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab10 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
    check("lab09 CUDA binary",
          remoteFileExists(env['HPC_LAB_DIR'].replace('lab10','lab09') + '/heat2Dcuda')),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> cluster] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab10 dir ready')


## Part 1 · Pin one GPU per rank

On a Polaris node with 4 A100s, if you launch 4 MPI ranks you want each rank to see one GPU. Pinning happens via `cudaSetDevice(localRank)`.

`localRank` is *rank within the node*, which MPI doesn't tell you directly. Compute it by splitting `MPI_COMM_WORLD` by hostname:

```c
MPI_Comm nodeComm;
MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, 0, MPI_INFO_NULL, &nodeComm);
int localRank; MPI_Comm_rank(nodeComm, &localRank);
cudaSetDevice(localRank);
```


In [ ]:
# [Hub] Skeleton for the pinning step.
print('At program start, right after MPI_Init:')
print('  MPI_Comm_split_type(...MPI_COMM_TYPE_SHARED...&nodeComm)')
print('  MPI_Comm_rank(nodeComm, &localRank)')
print('  cudaSetDevice(localRank)')


In [ ]:
checkpoint("Part 1 - pinning pattern", [
    check("lab dir ok", dirExists(str(labDir))),
])


## Part 2 · CUDA-aware MPI · pass device pointers directly

**Vanilla MPI** requires buffers to be in host memory. To send a device buffer, you'd `cudaMemcpy` to a host buffer, `MPI_Send`, and the receiver would `cudaMemcpy` back. Two copies per message.

**CUDA-aware MPI** (Cray MPICH on Polaris has this by default) lets you pass a device pointer directly to `MPI_Send`/`MPI_Isend`. The MPI runtime handles the transfer, potentially using GPUDirect RDMA to skip host memory entirely.

You don't have to write anything new; you just don't copy to host first.


In [ ]:
# [Hub -> Polaris] Confirm the MPI runtime is CUDA-aware. Load module,
# check compile-time flag.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
module load PrgEnv-nvhpc 2>/dev/null || true
echo === MPICH GTL info ===
cc -show 2>&1 | head -5
echo === MPICH environment for CUDA-aware ===
env | grep -i MPICH_GPU || echo "MPICH_GPU_SUPPORT_ENABLED may need to be set to 1 at job time"
'''
pbsPath = labDir/'chkJob.pbs'
pbsPath.write_text(pbsHeader(name='lab10Chk', project=env['HPC_PROJECT'],
                             queue='debug', select='1:system=polaris',
                             walltime='00:05:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/chk.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/chkJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/chkJob.pbs'); waitJob(jobID, 15, 600)
sshGet(env['HPC_LAB_DIR']+'/chk.out', str(labDir/'chk.out'))
print((labDir/'chk.out').read_text())


In [ ]:
checkpoint("Part 2 - MPI CUDA-aware check", [
    check("chk output", fileExists(str(labDir/'chk.out'))),
])


## Part 3 · Run on 4 GPUs, one node

This lab is deliberately smaller than lab 06/07: the goal is to prove the MPI+CUDA integration works on one Polaris node with 4 GPUs, not to do a big scaling study. Lab 11 does the scaling study across many nodes.


In [ ]:
# [Hub -> Polaris] Placeholder: use the CUDA binary from lab 09 wrapped in
# an mpiexec launcher. A full MPI+CUDA source is your semester project (lab 12).
showNote('Full MPI+CUDA heat2D source is your semester project (lab 12).\n'
         'This lab just proves the launch model works: 4 ranks x 1 GPU each.',
         kind='info')


In [ ]:
checkpoint("Part 3 - multi-gpu launch understood", [
    check("lab dir ok", dirExists(str(labDir))),
])


## Part 4 · NCCL as an alternative

For collective operations (allreduce, broadcast, allgather) between GPUs, [NCCL](https://developer.nvidia.com/nccl) is often faster than MPI. It understands GPU topology and can use NVLink where available. Most deep-learning frameworks (PyTorch, TensorFlow) use NCCL under the hood; classical HPC codes are just starting to.

For the heat stencil there's no collective in the hot path (halo exchange is point-to-point), so MPI is fine. If your project involves a reduction each step, benchmark both.


In [ ]:
checkpoint("Part 4 - NCCL awareness", [
    check("lab dir", dirExists(str(labDir))),
])


## Part 5 · Correctness

Multi-GPU with a proper halo exchange should agree on `sumU` to the same ~1e-8 tolerance as single-GPU. Any bigger disagreement points at a halo problem.


In [ ]:
checkpoint("Part 5 - correctness plan", [
    check("lab dir", dirExists(str(labDir))),
])


## Part 6 · Bridge to lab 11

You now have every parallel variant: serial (lab 01), OMP (lab 03), MPI (lab 06), hybrid (lab 07), GPU (lab 08), CUDA (lab 09), multi-GPU (this lab). Lab 11 is the **scaling study**: pick one variant, run it at 1, 2, 4, ..., N nodes/GPUs, plot strong and weak scaling, write up findings. This is the capstone that turns your semester of code into a report.


## Wrap up

Moved the spine forward one lab.


### Lab scorecard


In [ ]:
labSummary("Multi-GPU")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("Multi-GPU")
